# Jina-v4 textual + zerank-2 (ViDoRe V3)

Reproduce **Table 2** from the ViDoRe V3 paper: **Jina-v4 textual** retriever (text-only on document markdown) then **zerank-2** reranker. Physics / English.
Paper: Jina-v4 textual ~50.4 NDCG@10, + zerank-2 ~63.6.

**Dependencies:** `pip install mteb[jina-v4] sentence-transformers` (and optionally `flash-attn` for Jina-v4).

## Load data

In [ ]:
from datasets import load_dataset


def load_data_vidore(subset: str = "physics", lang: str = "english"):
    """Load ViDoRe v3 corpus, queries, qrels for a given subset and language."""
    name = f"vidore/vidore_v3_{subset}"
    ds_corpus = load_dataset(name, "corpus", split="test")
    ds_queries_full = load_dataset(name, "queries", split="test")
    ds_qrels_full = load_dataset(name, "qrels", split="test")
    ds_metadata = load_dataset(name, "documents_metadata", split="test")
    ds_queries = ds_queries_full.filter(lambda x: x["language"] == lang)
    query_ids = set(ds_queries["query_id"])
    ds_qrels = ds_qrels_full.filter(lambda x: x["query_id"] in query_ids)
    return ds_corpus, ds_queries, ds_qrels, ds_metadata


ds_corpus, ds_queries, ds_qrels, ds_metadata = load_data_vidore()
print(f"Queries: {len(ds_queries)}, Corpus: {len(ds_corpus)}")

In [ ]:
from collections import defaultdict

# qrels: query_id -> {corpus_id: score (0/1/2)}
relevants = defaultdict(dict)
for r in ds_qrels:
    relevants[r["query_id"]][r["corpus_id"]] = r["score"]

idx_to_corpus_id = [ds_corpus[i]["corpus_id"] for i in range(len(ds_corpus))]
query_id_to_query = {row["query_id"]: row["query"] for row in ds_queries}
# Document text (markdown) for textual retrieval
corpus_texts = [ds_corpus[i]["markdown"] or "" for i in range(len(ds_corpus))]

## Jina-v4 textual retriever

In [ ]:
import mteb
import numpy as np
from tqdm import tqdm

# Load Jina-v4 (text + image; we use text-only for "textual" pipeline)
jina = mteb.get_model("jinaai/jina-embeddings-v4", device="cuda")
jina_model = jina.model  # underlying HF model with encode_text

BATCH = 32

# Encode corpus (passages)
with __import__("torch").no_grad():
    corpus_emb = jina_model.encode_text(
        texts=corpus_texts,
        prompt_name="passage",
        task="retrieval",
        batch_size=BATCH,
        return_numpy=True,
    )
# Ensure numpy, float32
if hasattr(corpus_emb, "cpu"):
    corpus_emb = corpus_emb.cpu().float().numpy()
elif isinstance(corpus_emb, list):
    corpus_emb = np.stack([e.cpu().float().numpy() if hasattr(e, "cpu") else np.asarray(e) for e in corpus_emb])
corpus_emb = np.asarray(corpus_emb).astype(np.float32)
# L2 normalize for cosine = dot
corpus_emb = corpus_emb / (np.linalg.norm(corpus_emb, axis=1, keepdims=True) + 1e-9)
print("Corpus embeddings shape:", corpus_emb.shape)

In [ ]:
# Encode all queries (fixed order for reproducibility)
query_ids_ordered = sorted(relevants.keys())
queries_list = [query_id_to_query[qid] for qid in query_ids_ordered]

with __import__("torch").no_grad():
    query_emb = jina_model.encode_text(
        texts=queries_list,
        prompt_name="query",
        task="retrieval",
        batch_size=BATCH,
        return_numpy=True,
    )
if hasattr(query_emb, "cpu"):
    query_emb = query_emb.cpu().float().numpy()
elif isinstance(query_emb, list):
    query_emb = np.stack([e.cpu().float().numpy() if hasattr(e, "cpu") else np.asarray(e) for e in query_emb])
query_emb = np.asarray(query_emb).astype(np.float32)
query_emb = query_emb / (np.linalg.norm(query_emb, axis=1, keepdims=True) + 1e-9)
print("Query embeddings shape:", query_emb.shape)

In [ ]:
# Retrieve top-K per query (we'll rerank with zerank-2; paper uses reranker on top candidates)
RETRIEVAL_K = 100  # typical for reranking

scores = query_emb @ corpus_emb.T  # [n_queries, n_corpus]
top_k_indices = np.argsort(-scores, axis=1)[:, :RETRIEVAL_K]  # [n_queries, RETRIEVAL_K]

# For each query: top_k_indices[q] are corpus indices
print("Retrieval done. Shape per query:", top_k_indices.shape)

### (Optional) Jina-v4 textual only — NDCG@10 without reranker
Paper Table 2: Jina-v4 textual ~50.4.

In [ ]:
import math


def ndcg_at_k(relevance_at_rank: list, rel_dict: dict, k: int = 10) -> float:
    rel = relevance_at_rank[:k]
    dcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(rel))
    ideal = sorted(rel_dict.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


# Use first 10 from retrieval (no rerank)
ndcg_jina_only = []
for q_idx, qid in enumerate(query_ids_ordered):
    rel_dict = relevants[qid]
    top10_idx = top_k_indices[q_idx][:10].tolist()
    relevance_at_rank = [rel_dict.get(idx_to_corpus_id[i], 0) for i in top10_idx]
    ndcg_jina_only.append(ndcg_at_k(relevance_at_rank, rel_dict, k=10))

print(f"Physics / English / Jina-v4 textual only  NDCG@10 = {np.mean(ndcg_jina_only) * 100:.1f}")

## zerank-2 reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("zeroentropy/zerank-2", trust_remote_code=True)
RERANK_TOP = 10  # keep top 10 after rerank for NDCG@10

In [ ]:
# Rerank each query's top-K candidates and take top 10
def rerank_top_k(query: str, candidate_indices: list, corpus_texts: list, reranker, k: int = 10):
    pairs = [(query, corpus_texts[i]) for i in candidate_indices]
    scores = reranker.predict(pairs)
    order = np.argsort(-np.array(scores))
    return [candidate_indices[j] for j in order[:k]]

reranked_top10 = []
for q_idx in tqdm(range(len(query_ids_ordered)), desc="Rerank"):
    qid = query_ids_ordered[q_idx]
    query = query_id_to_query[qid]
    cand_indices = top_k_indices[q_idx].tolist()
    top10_idx = rerank_top_k(query, cand_indices, corpus_texts, reranker, k=RERANK_TOP)
    reranked_top10.append(top10_idx)

## NDCG@10

In [ ]:
import math


def ndcg_at_k(relevance_at_rank: list, rel_dict: dict, k: int = 10) -> float:
    """NDCG@k with graded relevance (ViDoRe: 0, 1, 2). IDCG from full rel_dict."""
    rel = relevance_at_rank[:k]
    dcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(rel))
    ideal = sorted(rel_dict.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


ndcg_scores = []
for q_idx, qid in enumerate(query_ids_ordered):
    rel_dict = relevants[qid]
    top10_idx = reranked_top10[q_idx]
    relevance_at_rank = [rel_dict.get(idx_to_corpus_id[i], 0) for i in top10_idx]
    ndcg_scores.append(ndcg_at_k(relevance_at_rank, rel_dict, k=10))

ndcg_mean = np.mean(ndcg_scores) * 100
print(f"Physics / English / Jina-v4 textual + zerank-2  NDCG@10 = {ndcg_mean:.1f}")
print("(Paper Table 2: ~63.6)")